<a href="https://colab.research.google.com/github/AKSHAYAVISWA/HexaTraining/blob/main/June%2017/Delta_Table_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install pyspark==3.5.0 delta-spark==3.1.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.8 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425346 sha256=3645aff4fad49f52efa368be569ef57baeb797bbe9cad252d5070a5a12fe92bb
  Stored in directory: /root/.cache/pip/wheels/84/40/20/65eefe766118e0a8f8e385cc3ed6e9eb7241c7e51cfc04c51a
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conne

In [5]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("DeltaTableExercises") \
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    ) \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [6]:
passengers_day1 = [
    (101,"Rahul Sharma","Hyderabad","Economy","India"),
    (102,"Priya Reddy","Bangalore","Business","India"),
    (103,"Amit Kumar","Mumbai","Economy","India"),
    (104,"Sneha Patel","Delhi","Premium Economy","India"),
    (105,"Farhan Ali","Chennai","Economy","India")
]

columns = [
    "passenger_id",
    "passenger_name",
    "city",
    "travel_class",
    "country"
]

df_day1 = spark.createDataFrame(
    passengers_day1,
    columns
)

df_day1.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [7]:
passengers_day2 = [
    (102,"Priya Reddy","Bangalore","First Class","India"),
    (104,"Sneha Patel","Hyderabad","Premium Economy","India"),
    (106,"Neha Singh","Pune","Economy","India"),
    (107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(
    passengers_day2,
    columns
)

df_day2.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [8]:
from delta.tables import DeltaTable

In [9]:
delta_path = "/tmp/passengers_delta"

In [10]:
df_day1.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

In [11]:
df_day1.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [16]:
delta_df = spark.read.format("delta").load(delta_path)

print("Record Count =", delta_df.count())

Record Count = 7


In [17]:
delta_df.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [18]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)

delta_table.history().show(truncate=False)

+-------+-----------------------+------+--------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+

In [19]:
delta_table.alias("target").merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Merge Completed")

Merge Completed


In [20]:
spark.read.format("delta").load(delta_path).show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [21]:
spark.read.format("delta").load(delta_path) \
    .filter("passenger_id IN (106,107)") \
    .show()

+------------+--------------+-----+------------+-------+
|passenger_id|passenger_name| city|travel_class|country|
+------------+--------------+-----+------------+-------+
|         106|    Neha Singh| Pune|     Economy|  India|
|         107|   Arjun Verma|Kochi|    Business|  India|
+------------+--------------+-----+------------+-------+



In [22]:
spark.read.format("delta").load(delta_path) \
    .filter("passenger_id = 102") \
    .show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [23]:
spark.read.format("delta").load(delta_path) \
    .filter("passenger_id = 106") \
    .show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [24]:
version0 = spark.read.format("delta") \
    .option("versionAsOf",0) \
    .load(delta_path)

version0.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [25]:
latest = spark.read.format("delta").load(delta_path)

latest.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [26]:
print("Version 0 Count :", version0.count())

print("Latest Count :", latest.count())

Version 0 Count : 5
Latest Count : 7


In [27]:
version0.filter("passenger_id = 102").show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+



In [28]:
latest.filter("passenger_id = 102").show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [29]:
version0.filter("passenger_id = 104").show()

+------------+--------------+-----+---------------+-------+
|passenger_id|passenger_name| city|   travel_class|country|
+------------+--------------+-----+---------------+-------+
|         104|   Sneha Patel|Delhi|Premium Economy|  India|
+------------+--------------+-----+---------------+-------+



In [30]:
latest.filter("passenger_id = 104").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+



In [31]:
spark.sql(f"""
OPTIMIZE delta.`{delta_path}`
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,totalClusterParallelism:bigint,totalScheduledTasks:bigint,autoCompactParallelismStats:struct<maxClusterActiveParallelism:bigint,minClusterActiveParallelism:bigint,maxSessionActiveParallelism:bigint,minSessionActiveParallelism:bigint>,de

In [32]:
spark.sql(f"""
OPTIMIZE delta.`{delta_path}`
ZORDER BY (city)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,totalClusterParallelism:bigint,totalScheduledTasks:bigint,autoCompactParallelismStats:struct<maxClusterActiveParallelism:bigint,minClusterActiveParallelism:bigint,maxSessionActiveParallelism:bigint,minSessionActiveParallelism:bigint>,de

In [33]:
delta_table.delete("passenger_id = 105")

print("Passenger 105 Deleted")

Passenger 105 Deleted


In [34]:
spark.read.format("delta").load(delta_path).show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [35]:
delta_table.history().show(truncate=False)

+-------+-----------------------+------+--------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------------------------

In [36]:
spark.sql(f"""
VACUUM delta.`{delta_path}` RETAIN 168 HOURS
""")

DataFrame[path: string]